**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Statistical Signal Processing

Real signals are random. This workshop supplies the theory the [adaptive filtering](../Intro_Time_Series/README.md) notebooks borrowed on credit: random processes and stationarity, spectral estimation done honestly, the Wiener filter derived, and matched filters & detection — the statistics of pulling signals out of noise.

## 1. Pre-requisites

- [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) & [Independence](../Intro_Math/Analysis/Independence.ipynb).
- [Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) (DFT, convolution).
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) helps for Session 4.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Random Processes & Stationarity* (~35 min)
**Goal:** treat a signal as a family of random variables; define autocorrelation and WSS.
**Builds on:** [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (spectral estimation).

---

## 2. Random Processes

💡 **Intuition.** A random process is a random variable *per time index* — one experiment produces a whole waveform (a *realization*). The process's personality lives in its joint statistics, but for signal processing two numbers usually suffice: the mean $\mu[n]$ and the **autocorrelation** $r[n, m] = E[x[n]x[m]]$ — how much the process remembers itself across time. **Wide-sense stationary (WSS)** means those two don't care about absolute time: $\mu$ constant, $r$ depends only on the lag $k = n - m$. Stationarity is what lets one long recording stand in for the whole ensemble (ergodicity — the [LLN](../Intro_Math/Analysis/Independence.ipynb) applied along time).

In [2]:
# Three processes, three memories: white, AR(1) smooth, AR(1) alternating
N, R = 2048, 400                       # length, realizations for ensemble averages
w = rng.standard_normal((R, N))
ar_pos = sig.lfilter([1], [1, -0.9], w, axis=1)    # remembers with + sign: smooth
ar_neg = sig.lfilter([1], [1, +0.9], w, axis=1)    # remembers with − sign: jittery

def acf_ensemble(X, maxlag=20):
    X = X - X.mean()
    return np.array([np.mean(X[:, :N-k] * X[:, k:]) for k in range(maxlag)])

fig, axes = plt.subplots(1, 2, figsize=(9.5, 2.8))
for name, X in [("white", w), ("AR(1) a=0.9", ar_pos), ("AR(1) a=−0.9", ar_neg)]:
    axes[0].plot(X[0, :200], alpha=0.7, label=name)
    r = acf_ensemble(X); axes[1].stem(np.arange(20), r / r[0], basefmt=" ", label=name) if name=="white" else axes[1].plot(r / r[0], "o-", alpha=0.7, label=name)
axes[0].set_title("one realization each"); axes[0].legend(fontsize=7)
axes[1].set_title("normalized autocorrelation r[k]/r[0]"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2027473/1597755380.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 4 — *The Power Spectral Density* (~40 min)
**Goal:** define the PSD; learn why the raw periodogram lies and how Welch fixes it.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Wiener).

---

## 3. The PSD

💡 **Intuition.** The PSD is the autocorrelation's Fourier transform (Wiener–Khinchin): it says how the process's *power* is distributed over frequency — the ensemble version of the spectrum. The trap: the **raw periodogram** $|X(\omega)|^2/N$ is an *inconsistent* estimator — its variance never shrinks, no matter how much data you record, because each frequency bin is essentially one squared Gaussian (~1 degree of freedom, χ²₂-fluctuating forever). **Welch's fix**: chop into segments, periodogram each, *average* — trading resolution for the variance decay the [LLN](../Intro_Math/Analysis/Independence.ipynb) provides.

In [3]:
# One AR(2) process, its TRUE spectrum, and two estimates from the SAME data
b_ar, a_ar = [1.0], [1.0, -1.2, 0.81]
x = sig.lfilter(b_ar, a_ar, rng.standard_normal(2**15))

f_true = np.linspace(0, 0.5, 512)
_, H = sig.freqz(b_ar, a_ar, worN=2*np.pi*f_true)
psd_true = np.abs(H)**2

f_p, P_raw = sig.periodogram(x)
f_w, P_welch = sig.welch(x, nperseg=512)

plt.figure(figsize=(9, 3))
plt.semilogy(f_p, P_raw, alpha=0.35, label="raw periodogram (variance never dies)")
plt.semilogy(f_w, P_welch, linewidth=2, label="Welch, 512-sample segments")
plt.semilogy(f_true, psd_true, "k--", linewidth=1.5, label="true PSD")
plt.xlim(0, 0.5); plt.legend(); plt.xlabel("normalized frequency")
plt.title("32768 samples: the periodogram still fuzzes, Welch converges")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2027473/481309448.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 4 — *The Wiener Filter, Derived* (~35 min)
**Goal:** solve the optimal linear filtering problem the adaptive filters approximate.
**Builds on:** Session 2; [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2. &nbsp; **Feeds into:** Session 4 (detection).

---

## 4. Optimal Linear Filtering

Problem: estimate desired $d[n]$ from observations $x[n]$ using an FIR filter $\hat{d} = \mathbf{w}^T \mathbf{x}[n]$, minimizing $E[e^2]$.

Setting the gradient to zero (the [matrix calculus](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) you've done) gives the **Wiener–Hopf equations**
$$R \mathbf{w}_o = \mathbf{p}, \qquad R = E[\mathbf{x}\mathbf{x}^T], \;\; \mathbf{p} = E[d \, \mathbf{x}],$$
— a projection ([orthogonality principle](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb): the optimal error is orthogonal to every observation). [LMS/APA](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) chase this solution without knowing $R, \mathbf{p}$; here we *compute* it.

💡 **Intuition.** In the frequency domain the noncausal solution is transparent: $W(\omega) = \frac{S_d(\omega)}{S_d(\omega) + S_v(\omega)}$ for signal-plus-noise — a **per-frequency trust dial** (compare the Kalman gain!). Where signal dominates, pass ≈ 1; where noise dominates, squash ≈ 0. Optimal filtering is spectral triage.

In [4]:
# Wiener denoising, built from PSDs alone
d = sig.lfilter([1], [1, -0.95], rng.standard_normal(2**14))   # smooth desired signal
v = 1.0 * rng.standard_normal(2**14)                            # white noise
x = d + v

f, S_d = sig.welch(d, nperseg=1024)
_, S_x = sig.welch(x, nperseg=1024)
W = S_d / (S_d + 1.0)                                           # unit-variance white noise: S_v = 1

# apply as zero-phase frequency-domain filter (block processing)
X = np.fft.rfft(x)
fW = np.interp(np.fft.rfftfreq(len(x)), f, W)
dhat = np.fft.irfft(X * fW, n=len(x))

print(f"input SNR  {10*np.log10(np.var(d)/np.var(v)):5.1f} dB")
print(f"output SNR {10*np.log10(np.var(d)/np.var(dhat - d)):5.1f} dB   (Wiener gain, no peeking at d's samples — only its PSD)")

input SNR   10.1 dB
output SNR  13.3 dB   (Wiener gain, no peeking at d's samples — only its PSD)


---
### 🕐 Session 4 of 4 — *Matched Filters & Detection* (~40 min)
**Goal:** detect a known pulse in noise optimally; meet ROC curves and the Neyman–Pearson view.
**Builds on:** Session 3; [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

---

## 5. Detection

💡 **Intuition.** Estimation asks 'what is $\theta$?'; detection asks '**is it there at all?**' For a known pulse in white Gaussian noise, the optimal detector correlates the data against the pulse — the **matched filter**, which is just the [Cauchy–Schwarz](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) statement that correlation against a template is maximized by the template itself. It maximizes SNR at the decision instant; radar, sonar, GPS, and your Wi-Fi preamble sync all run on it.

**Neyman–Pearson framing.** Choose the threshold to fix the false-alarm rate $P_{FA}$; the likelihood-ratio test (here: matched filter output vs threshold) then maximizes detection probability $P_D$. Sweeping the threshold traces the **ROC curve** — the universal report card of any detector, from radar to medical tests to spam filters.

In [5]:
# Matched filter vs naive energy detector, at SNR where it matters
pulse = sig.gausspulse(np.linspace(-1, 1, 64), fc=4)
pulse /= np.linalg.norm(pulse)
trials, sigma = 4000, 1.2

def run(with_pulse):
    x = sigma * rng.standard_normal((trials, 64))
    if with_pulse: x += pulse
    mf = x @ pulse                      # matched filter statistic
    en = (x**2).sum(1)                  # energy detector statistic
    return mf, en

mf1, en1 = run(True); mf0, en0 = run(False)

def roc(stat1, stat0):
    ths = np.quantile(np.concatenate([stat0, stat1]), np.linspace(0, 1, 200))
    return [( (stat0 > t).mean(), (stat1 > t).mean()) for t in ths]

plt.figure(figsize=(4.6, 4.2))
for name, (s1, s0) in [("matched filter", (mf1, mf0)), ("energy detector", (en1, en0))]:
    pts = np.array(roc(s1, s0))
    plt.plot(pts[:, 0], pts[:, 1], label=name)
plt.plot([0, 1], [0, 1], "k:", linewidth=0.8, label="coin flip")
plt.xlabel("$P_{FA}$"); plt.ylabel("$P_D$"); plt.legend()
plt.title("ROC: the matched filter dominates at every threshold")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2027473/1927027460.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Conclusion

WSS + ergodicity let one recording speak for the ensemble; Welch buys consistent spectra with the LLN; Wiener filtering is spectral triage and the target all adaptive filters chase; matched filtering + Neyman–Pearson is optimal 'is it there?'. This is the statistical spine of practical DSP.

---
## Where next

- [Adaptive Filtering](../Intro_Time_Series/Intro_AdFilt_APA.ipynb) — Wiener pursued online.
- [Array Processing](./Array_Processing.ipynb) — these tools across space, not just time.
- [Digital Communications](./Digital_Communications.ipynb) — matched filters earning rent every symbol.